In [74]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [75]:
data = pd.read_csv('AAPL.csv')
data.head(100)

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1980-12-12 00:00:00-05:00,0.098834,0.099264,0.098834,0.098834,469033600,0.0,0.0
1,1980-12-15 00:00:00-05:00,0.094108,0.094108,0.093678,0.093678,175884800,0.0,0.0
2,1980-12-16 00:00:00-05:00,0.087232,0.087232,0.086802,0.086802,105728000,0.0,0.0
3,1980-12-17 00:00:00-05:00,0.088951,0.089381,0.088951,0.088951,86441600,0.0,0.0
4,1980-12-18 00:00:00-05:00,0.091530,0.091959,0.091530,0.091530,73449600,0.0,0.0
...,...,...,...,...,...,...,...,...
95,1981-04-30 00:00:00-04:00,0.097545,0.098405,0.097545,0.097545,12611200,0.0,0.0
96,1981-05-01 00:00:00-04:00,0.097545,0.098405,0.097545,0.097545,16553600,0.0,0.0
97,1981-05-04 00:00:00-04:00,0.097545,0.097545,0.097116,0.097116,14448000,0.0,0.0
98,1981-05-05 00:00:00-04:00,0.097116,0.097116,0.096686,0.096686,17539200,0.0,0.0


In [76]:
data.dtypes

Date             object
Open            float64
High            float64
Low             float64
Close           float64
Volume            int64
Dividends       float64
Stock Splits    float64
dtype: object

In [77]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11079 entries, 0 to 11078
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          11079 non-null  object 
 1   Open          11079 non-null  float64
 2   High          11079 non-null  float64
 3   Low           11079 non-null  float64
 4   Close         11079 non-null  float64
 5   Volume        11079 non-null  int64  
 6   Dividends     11079 non-null  float64
 7   Stock Splits  11079 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 692.6+ KB


In [78]:
data['Date'] = pd.to_datetime(data['Date'], utc=True)

data['Quarter'] = data['Date'].dt.quarter




Calculating Features about data 

In [79]:
data['week_number'] = data['Date'].dt.isocalendar().week
data['week_day'] = data['Date'].dt.weekday
data['month']  = data['Date'].dt.month


calculating features on price 

In [86]:
#### Price Based Features #####

# Daily Range
data["Daily Range"] = data["High"] - data["Low"]

# Candle Size
data["Candle Size"] = (data["Close"] - data["Open"]).abs()

# 3. Direction
data["Direction"] = (data["Close"] > data["Open"]).astype(int)

# 4. Percentage Change
data["Percentage Change"] = ((data["Close"] - data["Open"]) / data["Open"]) * 100

# 5. Log Returns
data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))

# 6. Momentum (10-day)
data["Momentum_10"] = data["Close"] - data["Close"].shift(10)


# ===================== TREND AND MOVING AVERAGES =====================

# Simple Moving Average (SMA)
data["SMA_5"] = data["Close"].rolling(window=5).mean()
data["SMA_20"] = data["Close"].rolling(window=20).mean()

# Exponential Moving Average (EMA)
data["EMA_5"] = data["Close"].ewm(span=5, adjust=False).mean()
data["EMA_20"] = data["Close"].ewm(span=20, adjust=False).mean()


# ===================== VOLATILITY FEATURES =====================

# Rolling Standard Deviation (Volatility)
data["5-Day Volatility"] = data["Close"].rolling(window=5).std()


# Average True Range (ATR)
#data['TR'] = data[['High', 'Low', 'Close']].apply(
#    lambda row: max(row['High'] - row['Low'], 
#                    abs(row['High'] - row['Close'].shift(1)), 
#                    abs(row['Low'] - row['Close'].shift(1))) if not pd.isnull(row['Close'].shift(1)) else row['High'] - row['Low'], axis=1)
#data["ATR"] = data["TR"].rolling(window=14).mean()


# Bollinger Bands
data['Bollinger_SMA_20'] = data['Close'].rolling(window=20).mean()
data['Bollinger_Upper Band'] = data['Bollinger_SMA_20'] + (2 * data['Close'].rolling(window=20).std())
data['Bollinger_Lower Band'] = data['Bollinger_SMA_20'] - (2 * data['Close'].rolling(window=20).std())



# ===================== MOMENTUM AND OSCILLATORS =====================

# Relative Strength Index (RSI)
data['Change'] = data['Close'] - data['Close'].shift(1)
data['Gain'] = data['Change'].where(data['Change'] > 0, 0)
data['Loss'] = -data['Change'].where(data['Change'] < 0, 0)
data['Avg_Gain'] = data['Gain'].rolling(window=14).mean()
data['Avg_Loss'] = data['Loss'].rolling(window=14).mean()
data['RS'] = data['Avg_Gain'] / data['Avg_Loss']
data['RSI'] = 100 - (100 / (1 + data['RS']))



# Rate of Change (ROC)
data['ROC_10'] = ((data['Close'] - data['Close'].shift(10)) / data['Close'].shift(10)) * 100

# ===================== VOLUME-BASED FEATURES =====================

# Volume Moving Average
data['Volume_SMA_10'] = data['Volume'].rolling(window=10).mean()

# Volume Change (Percentage)
data['Volume Change'] = data['Volume'].pct_change() * 100

# On-Balance Volume (OBV)
data['OBV'] = (data['Volume'] * ((data['Close'] > data['Close'].shift(1)).astype(int) - (data['Close'] < data['Close'].shift(1)).astype(int))).cumsum()

# ===================== RETURN-BASED FEATURES =====================

# Daily Return
data['Daily Return'] = data['Close'].pct_change()

# Cumulative Return
data['Cumulative Return'] = (1 + data['Daily Return']).cumprod() - 1

# ===================== STATISTICAL FEATURES =====================

# Z-Score Normalization
data['Z-Score'] = (data['Close'] - data['Close'].rolling(window=20).mean()) / data['Close'].rolling(window=20).std()

# Skewness
data['Rolling Skewness'] = data['Close'].rolling(window=20).skew()

# Kurtosis
data['Rolling Kurtosis'] = data['Close'].rolling(window=20).kurt()

In [88]:
# 2. Calculate returns over different periods
data['7-Day Return'] = (data['Close'] / data['Close'].shift(7) - 1) * 100  # 7 days
data['1-Month Return'] = (data['Close'] / data['Close'].shift(30) - 1) * 100  # 30 days
data['6-Month Return'] = (data['Close'] / data['Close'].shift(182) - 1) * 100  # ~6 months
data['1-Year Return'] = (data['Close'] / data['Close'].shift(365) - 1) * 100  # 1 year

# 3. Calculate total return (all-time return from the first date)
data['Total Return'] = (data['Close'] / data['Close'].iloc[0] - 1) * 100

In [98]:
row_1000 = data.iloc[1000]

print(row_1000)



Date                    1984-11-26 05:00:00+00:00
Open                                     0.082506
High                                     0.082506
Low                                      0.082506
Close                                    0.082506
Volume                                  100643200
Dividends                                     0.0
Stock Splits                                  0.0
Quarter                                         4
week_number                                    48
week_day                                        0
month                                          11
Daily Range                                   0.0
Candle Size                                   0.0
Direction                                       0
Percentage Change                             0.0
Log Return                               0.010471
Momentum_10                              0.002578
SMA_5                                    0.079326
SMA_20                                   0.083021
